# Iris Mage pipeline: recorded assignment evidence

**Evidence status:** Every result below is reproduced from saved Mage logs, reports, existing block source, and pipeline metadata. This notebook was not used to execute Mage. Captured log excerpts are evidence from prior runs, not fresh notebook outputs. No instructor notebook in `reference/` was changed.

## Pipeline graph and six blocks

The existing metadata at [`pipelines/iris_classifier/metadata.yaml`](pipelines/iris_classifier/metadata.yaml) describes `load_iris → validate_iris → prepare_data → train_model`, with `train_model` branching to `save_results` and `evaluate_model`. The branch has one `evaluate_model` block.

| Block | Mage type | Test purpose (source) |
|---|---|---|
| `load_iris` | `data_loader` | `test_output`: 150 observations and a `target` column. |
| `validate_iris` | `transformer` | `test_output`: no missing values and exactly three target classes. |
| `prepare_data` | `transformer` | `test_output`: 120 train and 30 test rows (80/20 split). |
| `train_model` | `transformer` | `test_accuracy`: held-out accuracy at least 0.8. |
| `save_results` | `data_exporter` | `test_output`: predictions column exists and 30 rows exported. |
| `evaluate_model` | `transformer` | `test_output`: expected metric names/range and valid 3×3 confusion matrix CSV. |

Block sources: [`data_loaders/load_iris.py`](data_loaders/load_iris.py), [`transformers/validate_iris.py`](transformers/validate_iris.py), [`transformers/prepare_data.py`](transformers/prepare_data.py), [`transformers/train_model.py`](transformers/train_model.py), [`data_exporters/save_results.py`](data_exporters/save_results.py), [`transformers/evaluate_model.py`](transformers/evaluate_model.py).

## C1 — actual pipeline run

Source: [`../reports/c1_mage_run.log`](../reports/c1_mage_run.log). The captured Docker Mage command completed with exit code 0; output records six `1/1 tests passed.` results and `Pipeline run completed.` The log also prints the depth-3 confusion matrix and metrics reproduced in C3 below.

```text
1/1 tests passed.
1/1 tests passed.
1/1 tests passed.
1/1 tests passed.
1/1 tests passed.
1/1 tests passed.
Pipeline run completed.
C1_EXIT_CODE=0
```

This excerpt is transcribed from the saved log; the surrounding warning and evaluation output remain in the source log.

## C2 — actual max-depth experiments

Source: [`../reports/c2_depth_experiments.md`](../reports/c2_depth_experiments.md) and individual [`depth 1`](../reports/c2_depth_1.log), [`depth 2`](../reports/c2_depth_2.log), [`depth 3`](../reports/c2_depth_3.log), [`depth 4`](../reports/c2_depth_4.log), [`depth 5`](../reports/c2_depth_5.log) logs. Each was a Mage Docker run with `--runtime-vars`, and `iris_predictions.csv` was removed before the run.

| `max_depth` | Recorded accuracy | `train_model` test | Mage exit | CSV created |
|---:|---:|---|---:|---|
| 1 | 0.633 | FAIL (below 0.8) | 1 | No |
| 2 | 0.966667 (29/30) | PASS | 0 | Yes |
| 3 | 1.000 (30/30) | PASS | 0 | Yes |
| 4 | 1.000 (30/30) | PASS | 0 | Yes |
| 5 | 1.000 (30/30) | PASS | 0 | Yes |

At depth 1 Mage's test reported `Model accuracy 0.633 is below 0.8`. The test is a gate on the `train_model` block: when it fails, Mage marks that block failed and does not schedule its downstream `save_results` exporter, so the CSV is absent. Checking the threshold before export prevents a model that fails its acceptance criterion from being published as usable output; post-export checking would leave a bad artifact to detect and clean up.

## C3 — actual evaluation

Sources: [`../reports/c3_evaluation.md`](../reports/c3_evaluation.md), [`../reports/c3_mage_run.log`](../reports/c3_mage_run.log), [`../reports/c3_confusion_matrix.csv`](../reports/c3_confusion_matrix.csv). Actual depth-3 run exited 0; all six block tests passed.

Recorded confusion matrix (rows = actual, columns = predicted; label order `[0, 1, 2]`):

| Actual \ Predicted | 0 | 1 | 2 |
|---:|---:|---:|---:|
| 0 | 10 | 0 | 0 |
| 1 | 0 | 9 | 0 |
| 2 | 0 | 0 | 11 |

Recorded metric output:

| Metric | Value |
|---|---:|
| Accuracy | 1.0 |
| Precision (macro) | 1.0 |
| Recall (macro) | 1.0 |
| F1 score (macro) | 1.0 |

The saved matrix CSV is `actual_class,0,1,2` followed by rows `0,10,0,0`, `1,0,9,0`, and `2,0,0,11`.

## C4 — custom validation block

Sources: [`../reports/c4_custom_block.md`](../reports/c4_custom_block.md), [`../reports/c4_mage_run.log`](../reports/c4_mage_run.log), and [`transformers/validate_iris.py`](transformers/validate_iris.py). `validate_iris` is the `transformer` directly after `load_iris` and before `prepare_data`; it returns the original dataframe unchanged after checking no missing values and exactly three target classes. Its own `@test` asserts both properties. The captured C4 run exited 0 and reports six block tests passing; Mage's log prints generic test counts, so the validation test is identified by its source and graph position rather than by an invented named log line.

## Evidence references

The actual Mage execution logs and reports are under `../reports/`. The six listed block files and `pipelines/iris_classifier/metadata.yaml` are the existing project implementation. This notebook is a student-facing evidence summary, not a replacement for raw logs or source code.

In [1]:
from pathlib import Path
import os
from IPython.display import Markdown, display

def _find_project_root():
    anchors = ("reports/assignment_report.md", "elt/freshcart_elt.ipynb", "iris_project/pipelines/iris_classifier/metadata.yaml")
    starts = []
    configured = os.environ.get("FRESHCART_PROJECT_ROOT")
    if configured:
        starts.append(Path(configured).expanduser())
    if globals().get("__file__"):
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd().resolve())
    visited = set()
    for start in starts:
        for candidate in (start, *start.parents):
            candidate = candidate.resolve()
            if candidate in visited:
                continue
            visited.add(candidate)
            if all((candidate / anchor).is_file() for anchor in anchors):
                return candidate
    raise FileNotFoundError("Could not locate the project root from FRESHCART_PROJECT_ROOT, the notebook file, or the current directory.")

PROJECT_ROOT = _find_project_root()
REPORTS_DIR = PROJECT_ROOT / "reports"
display(Markdown('## Captured Mage execution evidence (read-only)\n\nThe outputs below are rendered from saved reports, logs, and CSV files. They are **not fresh Mage executions**. These cells only read local evidence files.'))


## Captured Mage execution evidence (read-only)

The outputs below are rendered from saved reports, logs, and CSV files. They are **not fresh Mage executions**. These cells only read local evidence files.

In [2]:
for filename in ("c2_depth_experiments.md", "c3_evaluation.md", "c4_custom_block.md"):
    evidence_path = REPORTS_DIR / filename
    if not evidence_path.is_file():
        raise FileNotFoundError(evidence_path)
    display(Markdown(f"## Captured evidence: {filename}\n\nSource file: `{evidence_path.relative_to(PROJECT_ROOT).as_posix()}`\n\n---\n\n" + evidence_path.read_text(encoding="utf-8")))


## Captured evidence: c2_depth_experiments.md

Source file: `reports/c2_depth_experiments.md`

---

# Part C2: Mage max-depth experiments

All five experiments ran sequentially in the official `mageai/mageai:latest` Docker image using the existing `iris_project`. Before each run, `iris_predictions.csv` was removed from the project root. The accuracy for the failed run comes from Mage's `test_accuracy` assertion; successful-run accuracy was calculated from that run's CSV by comparing `target` and `predicted` across its 30 rows.

| `max_depth` | Actual accuracy | `train_model` test | Mage exit code | CSV created |
|---:|---:|---|---:|---|
| 1 | 0.633 | FAIL (below 0.8) | 1 | No |
| 2 | 0.966667 (29/30) | PASS | 0 | Yes |
| 3 | 1.000 (30/30) | PASS | 0 | Yes |
| 4 | 1.000 (30/30) | PASS | 0 | Yes |
| 5 | 1.000 (30/30) | PASS | 0 | Yes |

## Why depth 1 has no prediction CSV

Mage ran `train_model`'s `test_accuracy` against its output and reported `Model accuracy 0.633 is below 0.8`. Because that test failed, Mage marked the `train_model` block as failed and did not proceed to its downstream `save_results` exporter. The CSV was absent after the run. This is the expected depth 1 result; the model logic and test threshold were not changed.

## Actual execution evidence

Each linked log contains the complete captured Mage stdout and stderr for that invocation:

- [Depth 1 Mage log](c2_depth_1.log) — assertion: `Model accuracy 0.633 is below 0.8`; Mage exit code 1; CSV absent.
- [Depth 2 Mage log](c2_depth_2.log) — `Pipeline run completed.`; Mage exit code 0; CSV created (30 rows).
- [Depth 3 Mage log](c2_depth_3.log) — `Pipeline run completed.`; Mage exit code 0; CSV created (30 rows).
- [Depth 4 Mage log](c2_depth_4.log) — `Pipeline run completed.`; Mage exit code 0; CSV created (30 rows).
- [Depth 5 Mage log](c2_depth_5.log) — `Pipeline run completed.`; Mage exit code 0; CSV created (30 rows).

Command template used for each run (the JSON depth was changed for each experiment):

```powershell
docker run --rm -v "${PWD}:/home/src" -w /home/src mageai/mageai:latest mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": N}'
```



## Captured evidence: c3_evaluation.md

Source file: `reports/c3_evaluation.md`

---

# Part C3: Iris evaluation

## Pipeline branch

The existing pipeline metadata contains exactly one `evaluate_model` block. `train_model` has exactly two downstream blocks, `save_results` and `evaluate_model`. No duplicate block was added and the pipeline structure was not rebuilt.

## Depth 3 execution results

Command run in `mageai/mageai:latest`:

```text
mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 3}'
```

The final run exited with code 0. All six block tests passed, and Mage reported `Pipeline run completed.` The evaluation output was:

```text
Confusion matrix (rows = actual, columns = predicted; labels = [0, 1, 2]):
    0  1   2
0  10  0   0
1   0  9   0
2   0  0  11
Evaluation metrics:
   metric  value
 accuracy    1.0
precision    1.0
   recall    1.0
 f1_score    1.0
```

The generated [confusion matrix CSV](c3_confusion_matrix.csv) contains:

```csv
actual_class,0,1,2
0,10,0,0
1,0,9,0
2,0,0,11
```

Complete successful Mage output: [c3_mage_run.log](c3_mage_run.log).

The first implementation attempt stored the matrix in `DataFrame.attrs`; Mage's output serialization did not preserve those attributes for the test, and that run failed with `KeyError`. The test now validates the persisted CSV directly. The first attempt's complete log is retained at [c3_mage_run_initial_test_failure.log](c3_mage_run_initial_test_failure.log); the final run above passed.

## Pipeline metadata used

This is the existing `iris_project/pipelines/iris_classifier/metadata.yaml` content used for the successful run:

```yaml
blocks:
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks: [validate_iris]
- name: validate_iris
  uuid: validate_iris
  type: transformer
  language: python
  upstream_blocks: [load_iris]
  downstream_blocks: [prepare_data]
- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks: [validate_iris]
  downstream_blocks: [train_model]
- name: train_model
  uuid: train_model
  type: transformer
  language: python
  upstream_blocks: [prepare_data]
  downstream_blocks: [save_results, evaluate_model]
- name: save_results
  uuid: save_results
  type: data_exporter
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
- name: evaluate_model
  uuid: evaluate_model
  type: transformer
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
name: iris_classifier
uuid: iris_classifier
type: python
variables:
  max_depth: 3
```

## Mage UI and temporary tunnel

Mage UI is running in Docker on port 6789 and returned HTTP 200 locally. The container mounts only `iris_project` at `/home/src/iris_project`; it does not mount the repository root or `.env`. Mage's startup log reported that user authentication is enabled.

`cloudflared` was not installed or on PATH in Windows. A temporary Cloudflare Quick Tunnel was started in a separate Docker container, without credentials or configuration mounts, and returned HTTP 200 when checked. Its temporary public URL is intentionally omitted here. The tunnel was only for the assignment screenshot; no credentials were exposed through it.

To stop the tunnel and Mage UI containers:

```powershell
docker stop iris-cloudflared-c3 iris-mage-ui-c3
docker rm iris-cloudflared-c3 iris-mage-ui-c3
```

To install `cloudflared` on Windows later, download the Windows MSI or executable from [Cloudflare's official downloads page](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/downloads/), install it, and reopen PowerShell so the updated PATH is loaded. A Quick Tunnel can then be started with `cloudflared tunnel --url http://localhost:6789`; it is temporary and requires no credentials or account.

The raw tunnel log is excluded from submission because it contains the temporary public URL.


## Captured evidence: c4_custom_block.md

Source file: `reports/c4_custom_block.md`

---

# Part C4: `validate_iris` verification

## Verification

All five requested requirements are already met; no block logic or pipeline metadata was changed.

1. `validate_iris` is a `transformer` block.
2. It follows `load_iris` directly in the graph.
3. It raises `ValueError` if any value is missing.
4. It raises `ValueError` unless `target` contains exactly three unique classes, and returns the input dataframe unchanged when valid.
5. Its own `@test` checks both that the output has no missing values and that it contains exactly three target classes.

## Existing block code

```python
@transformer
def validate(df: pd.DataFrame, *args, **kwargs) -> pd.DataFrame:
    """Return the data unchanged after checking its basic structure."""
    if df.isna().any().any():
        raise ValueError("Iris data contains missing values")
    if df["target"].nunique() != 3:
        raise ValueError("Iris data must contain exactly three classes")
    return df


@test
def test_output(output: pd.DataFrame, *args) -> None:
    assert not output.isna().any().any()
    assert output["target"].nunique() == 3
```

The block uses conditional imports for Mage's `transformer` and `test` decorators, preserving importability when Mage is absent.

## Position in `metadata.yaml`

The existing graph places the single `validate_iris` block between `load_iris` and `prepare_data`:

```yaml
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks: [validate_iris]
- name: validate_iris
  uuid: validate_iris
  type: transformer
  language: python
  upstream_blocks: [load_iris]
  downstream_blocks: [prepare_data]
- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks: [validate_iris]
  downstream_blocks: [train_model]
```

There is one `validate_iris` UUID/name in the pipeline metadata; no duplicate block was created.

## Mage execution evidence

Command run inside Docker:

```text
docker run --rm -v "${PWD}:/home/src" -w /home/src mageai/mageai:latest mage run iris_project iris_classifier --test
```

Mage exited with code 0 and reported six block tests passing and `Pipeline run completed.` The log emits each test result as `1/1 tests passed.` without a block name; in graph execution order, `validate_iris` is the second result, after `load_iris` and before `prepare_data`.

Complete captured output: [c4_mage_run.log](c4_mage_run.log).


In [3]:
for filename in ("c1_mage_run.log", "c3_mage_run.log", "c4_mage_run.log"):
    evidence_path = REPORTS_DIR / filename
    if not evidence_path.is_file():
        raise FileNotFoundError(evidence_path)
    print(f"\n===== CAPTURED LOG: {evidence_path.relative_to(PROJECT_ROOT).as_posix()} (read-only; prior Mage run) =====")
    print(evidence_path.read_text(encoding="utf-8"))



===== CAPTURED LOG: reports/c1_mage_run.log (read-only; prior Mage run) =====
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix (rows = actual, columns = predicted; labels = [0, 1, 2]):
    0  1   2
0  10  0   0
1   0  9   0
2   0  0  11
Evaluation metrics:
   metric  value
 accuracy    1.0
precision    1.0
   recall    1.0
 f1_score    1.0
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.
C1_EXIT_CODE=0


===== CAPTURED LOG: reports/c3_mage_run.log (read-

In [4]:
# This evidence display uses pandas without PyArrow; skip an unrelated optional PyArrow import.
import sys
sys.modules["pyarrow"] = None
import pandas as pd
from IPython.display import display, Markdown, HTML
from io import StringIO

c2_path = REPORTS_DIR / "c2_depth_experiments.md"
csv_path = REPORTS_DIR / "c3_confusion_matrix.csv"
if not c2_path.is_file() or not csv_path.is_file():
    raise FileNotFoundError("A cited C2/C3 evidence file is missing.")
lines = c2_path.read_text(encoding="utf-8").splitlines()
table_start = next(i for i, line in enumerate(lines) if "max_depth" in line and line.startswith("|"))
table_lines = []
for line in lines[table_start:]:
    if not line.startswith("|"):
        break
    table_lines.append(line)
parsed = [[part.strip().strip("`") for part in line.strip().strip("|").split("|")] for line in table_lines]
headers = parsed[0]
rows = [row for row in parsed[1:] if row and not all(set(value) <= set("-:") for value in row)]
results_df = pd.DataFrame(rows, columns=headers)
display(Markdown("## C2 recorded max-depth results (parsed from the saved report; not rerun)"))
display(HTML(results_df.to_html(index=False)))
confusion_df = pd.read_csv(csv_path, index_col=0)
display(Markdown("## C3 recorded confusion matrix (loaded from the saved CSV; labels are actual classes)"))
display(HTML(confusion_df.to_html()))


## C2 recorded max-depth results (parsed from the saved report; not rerun)

max_depth,Actual accuracy,train_model` test,Mage exit code,CSV created
1,0.633,FAIL (below 0.8),1,No
2,0.966667 (29/30),PASS,0,Yes
3,1.000 (30/30),PASS,0,Yes
4,1.000 (30/30),PASS,0,Yes
5,1.000 (30/30),PASS,0,Yes


## C3 recorded confusion matrix (loaded from the saved CSV; labels are actual classes)

,0,1,2
actual_class,,,
0,10,0,0
1,0,9,0
2,0,0,11
